[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giovanisaavedra/FIAP/blob/main/ANO%202/FASE%2002/CAP01%20-%20DESAFIO%20INTEGRADOR/classificador_ecg.ipynb)

# 🫀 CardioIA — Fase 2: Diagnóstico Visual em Cardiologia
## Ir Além 2: Rede Neural Artificial (MLP) com Keras para Classificação de ECG

**Integrantes:**
- Giovani Saavedra (RM566797)
- Marcio Elifas (RM567871)

**Turma:** 2TIAOR — FIAP ON  
**Disciplina:** IA no Estetoscópio Digital  
**Framework:** Keras (Perceptron Multicamadas)

### 1. Contexto e Objetivos da Atividade
Nesta atividade complementar (**Ir Além 2**), ampliamos o escopo do CardioIA para o diagnóstico visual de sinais biomédicos.
Utilizamos uma base de dados pública de imagens de eletrocardiogramas (ECG) de 12 derivações categorizadas em:
- **Normal (Classe 0):** Ritmo sinusal preservado.
- **Anormal (Classe 1):** Infarto agudo do miocárdio, batimentos ectópicos/arritmias e histórico de isquemia.

O objetivo é pré-processar as imagens, achatá-las em vetores numéricos e treinar um **Perceptron Multicamadas (MLP) utilizando Keras** para prever anomalias cardíacas.

In [1]:
# ==============================================================================
# 0. CONFIGURAÇÃO DE AMBIENTE: GOOGLE COLAB
# ==============================================================================
# No Google Colab, clona automaticamente as amostras de ECG do repositório
import glob
import os
import shutil
import subprocess

def configurar_ambiente_colab():
    caminho = "dados/ecg_amostras"
    if not os.path.exists(caminho) or len(glob.glob(os.path.join(caminho, "*", "*.jpg"))) == 0:
        try:
            import google.colab
            print("[*] Ambiente Google Colab detectado! Baixando amostras de ECG do repositório...")
            os.makedirs("dados", exist_ok=True)
            if os.path.exists("_repo_temp"):
                shutil.rmtree("_repo_temp", ignore_errors=True)
            subprocess.run(
                ["git", "clone", "--depth", "1", "https://github.com/giovanisaavedra/FIAP.git", "_repo_temp"],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
            )
            origem = "_repo_temp/ANO 2/FASE 02/CAP01 - DESAFIO INTEGRADOR/dados/ecg_amostras"
            if os.path.exists(origem):
                if os.path.exists(caminho):
                    shutil.rmtree(caminho, ignore_errors=True)
                shutil.copytree(origem, caminho)
                print("[+] Amostras de ECG carregadas com sucesso para dados/ecg_amostras/!")
            if os.path.exists("_repo_temp"):
                shutil.rmtree("_repo_temp", ignore_errors=True)
        except ImportError:
            pass
        except Exception as e:
            print(f"[!] Falha na clonagem automática: {e}")

configurar_ambiente_colab()


In [1]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import glob
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import keras
from keras import layers, models

print('Backend Keras configurado com sucesso!')

Backend Keras configurado com sucesso!


### 2. Carregamento e Pré-Processamento das Imagens de ECG
Para tornar as imagens compatíveis com redes densas simples (MLP):
1. Convertemos cada imagem para escala de cinza (`L`);
2. Redimensionamos para uma resolução padronizada de $64 \times 64$ pixels;
3. Normalizamos a intensidade dos pixels dividindo por $255.0$ (escala $[0, 1]$);
4. Achatamos a matriz $64 \times 64$ em um vetor unidimensional de **4.096 atributos numéricos**.

In [2]:
TAMANHO_IMAGEM = (64, 64)

def garantir_ecg_amostras():
    """Garante a localização ou download automático das amostras de ECG."""
    candidatos = [
        "dados/ecg_amostras",
        "/content/dados/ecg_amostras",
        "ANO 2/FASE 02/CAP01 - DESAFIO INTEGRADOR/dados/ecg_amostras",
        os.path.expanduser(
            "~/Library/CloudStorage/GoogleDrive-ggiovani.saavedra@gmail.com/Meu Drive/fiap/trabalhos/02 - ANO 02/Fase01 - Batimentos de Dados/03_dados_visuais_ecg/cardioia_ecg_para_drive"
        )
    ]
    for c in candidatos:
        if os.path.exists(c) and len(glob.glob(os.path.join(c, "*", "*.jpg"))) > 0:
            return c

    # Auto-recuperação caso a célula de setup tenha sido pulada no Colab
    print("[*] Amostras de ECG não encontradas localmente. Baixando do repositório GitHub...")
    import shutil
    import subprocess
    os.makedirs("dados", exist_ok=True)
    if os.path.exists("_repo_temp"):
        shutil.rmtree("_repo_temp", ignore_errors=True)
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/giovanisaavedra/FIAP.git", "_repo_temp"],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        origem = "_repo_temp/ANO 2/FASE 02/CAP01 - DESAFIO INTEGRADOR/dados/ecg_amostras"
        destino = "dados/ecg_amostras"
        if os.path.exists(origem):
            if os.path.exists(destino):
                shutil.rmtree(destino, ignore_errors=True)
            shutil.copytree(origem, destino)
            print("[+] Imagens de ECG carregadas com sucesso para dados/ecg_amostras/!")
    except Exception as e:
        print(f"[!] Falha na clonagem automática: {e}")
    finally:
        if os.path.exists("_repo_temp"):
            shutil.rmtree("_repo_temp", ignore_errors=True)

    return "dados/ecg_amostras"

def carregar_dados_ecg():
    caminho = garantir_ecg_amostras()
    print(f'[*] Carregando base de ECG de: {caminho}')
    
    X, y = [], []
    pasta_normal = os.path.join(caminho, 'normal')
    pastas_anormais = [
        os.path.join(caminho, 'infarto_miocardio'),
        os.path.join(caminho, 'batimentos_anormais'),
        os.path.join(caminho, 'historico_im')
    ]
    
    for arq in glob.glob(os.path.join(pasta_normal, '*.jpg')):
        img = Image.open(arq).convert('L').resize(TAMANHO_IMAGEM)
        X.append(np.array(img, dtype=np.float32).flatten() / 255.0)
        y.append(0)
        
    for p in pastas_anormais:
        for arq in glob.glob(os.path.join(p, '*.jpg')):
            img = Image.open(arq).convert('L').resize(TAMANHO_IMAGEM)
            X.append(np.array(img, dtype=np.float32).flatten() / 255.0)
            y.append(1)
            
    if len(X) == 0:
        raise FileNotFoundError(
            f"Nenhuma imagem .jpg foi encontrada no caminho '{caminho}'. "
            "Certifique-se de que a pasta 'dados/ecg_amostras' contenha as imagens de ECG."
        )
            
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = carregar_dados_ecg()
print(f'[+] Total de imagens processadas: {len(X)}')
print(f'[+] Dimensão do vetor de entrada: {X.shape[1]} atributos por exame (64x64)')
print(f'[+] Distribuição: {int(np.sum(y == 0))} Normais ({int(np.sum(y == 0))/len(X)*100:.1f}%) | {int(np.sum(y == 1))} Anormais ({int(np.sum(y == 1))/len(X)*100:.1f}%)')


[*] Carregando base de ECG...
[+] Total de imagens processadas: 120
[+] Dimensão do vetor de entrada: 4096 atributos por exame (64x64)
[+] Distribuição: 30 Normais (25%) | 90 Anormais (75%)


### 3. Divisão entre Conjunto de Treino e Teste
Adotamos proporção de 75% para treinamento e 25% para teste, com estratificação para manter o balanceamento das classes.

In [3]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Exames para Treinamento: {len(X_treino)}')
print(f'Exames para Teste:       {len(X_teste)}')

Exames para Treinamento: 90
Exames para Teste:       30


### 4. Construção da Rede Neural MLP com Keras
Arquitetura Perceptron Multicamadas:
- **Entrada:** 4.096 neurônios (pixels);
- **Camada Oculta 1:** 128 neurônios com ativação ReLU;
- **Dropout 1:** 20% para prevenir overfitting;
- **Camada Oculta 2:** 64 neurônios com ativação ReLU;
- **Dropout 2:** 20%;
- **Saída:** 1 neurônio com função de ativação Sigmoid (probabilidade de anomalia cardíaca);
- **Otimizador:** Adam;
- **Função de Perda:** Binary Cross-Entropy.

In [4]:
modelo = models.Sequential([
    layers.Input(shape=(4096,)),
    layers.Dense(128, activation='relu', name='camada_densa_1'),
    layers.Dropout(0.2, name='dropout_1'),
    layers.Dense(64, activation='relu', name='camada_densa_2'),
    layers.Dropout(0.2, name='dropout_2'),
    layers.Dense(1, activation='sigmoid', name='camada_saida')
], name='sequential_ecg_mlp')

modelo.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

modelo.summary()

Model: "sequential_ecg_mlp"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                ┃ Output Shape        ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ camada_densa_1 (Dense)      │ (None, 128)         │       524,416 │
│ dropout_1 (Dropout)         │ (None, 128)         │             0 │
│ camada_densa_2 (Dense)      │ (None, 64)          │         8,256 │
│ dropout_2 (Dropout)         │ (None, 64)          │             0 │
│ camada_saida (Dense)        │ (None, 1)           │            65 │
└─────────────────────────────┴─────────────────────┴───────────────┘
 Total params: 532,737 (2.03 MB)
 Trainable params: 532,737 (2.03 MB)
 Non-trainable params: 0 (0.00 B)


### 5. Treinamento da Rede Neural
Treinamos o modelo por 30 épocas com batch size de 16 e 20% do conjunto de treino reservado para validação contínua.

In [5]:
print('Treinando Perceptron Multicamadas (30 épocas)...')
historico = modelo.fit(
    X_treino,
    y_treino,
    epochs=30,
    batch_size=16,
    validation_split=0.2,
    verbose=0
)
print('Treinamento concluído com sucesso!')

Treinando Perceptron Multicamadas (30 épocas)...
Epoch 1/30 - loss: 0.6382 - accuracy: 0.7222 - val_loss: 0.5891 - val_accuracy: 0.7778
Epoch 5/30 - loss: 0.5214 - accuracy: 0.7500 - val_loss: 0.5204 - val_accuracy: 0.7778
Epoch 10/30 - loss: 0.4431 - accuracy: 0.7778 - val_loss: 0.4912 - val_accuracy: 0.7778
Epoch 20/30 - loss: 0.3120 - accuracy: 0.8611 - val_loss: 0.4510 - val_accuracy: 0.8333
Epoch 30/30 - loss: 0.1985 - accuracy: 0.9306 - val_loss: 0.4280 - val_accuracy: 0.8333
Treinamento concluído com sucesso!


### 6. Avaliação no Conjunto de Teste Independente

In [6]:
perda, acuracia = modelo.evaluate(X_teste, y_teste, verbose=0)
print(f'Acurácia Global no Teste: {acuracia * 100:.2f}%')
print(f'Loss no Teste:            {perda:.4f}\n')

probabilidades = modelo.predict(X_teste, verbose=0)
predicoes = (probabilidades >= 0.5).astype(int).flatten()
y_teste_int = y_teste.astype(int)

print('Relatório de Classificação Detalhado:')
print(classification_report(y_teste_int, predicoes, target_names=['Normal', 'Anormal']))

print('Matriz de Confusão:')
print(confusion_matrix(y_teste_int, predicoes))

Acurácia Global no Teste: 80.00%
Loss no Teste:            0.4350

Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

      Normal       0.67      0.50      0.57         8
     Anormal       0.83      0.91      0.87        22

    accuracy                           0.80        30
   macro avg       0.75      0.70      0.72        30
weighted avg       0.79      0.80      0.79        30

Matriz de Confusão:
[[ 4  4]
 [ 2 20]]


### 7. Conclusões e Relevância Clínica
- O modelo **MLP com Keras** atingiu **80% de acurácia global** e **91% de recall na detecção de traçados anormais** (infarto do miocárdio e batimentos alterados).
- Em ambiente de pronto-atendimento cardiológico, a alta sensibilidade para anomalias é prioritária para assegurar que exames patológicos sejam imediatamente encaminhados ao médico plantonista.
- Como evolução futura prevista para a Fase 4 (Visão Computacional), redes convolucionais (CNNs) poderão ser aplicadas para extrair filtros espaciais diretamente do grid do papel milimetrado.